In [78]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [79]:
!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

'1.2.2'

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from sklearn.metrics import davies_bouldin_score
from collections import Counter


In [81]:
pypfopt.__version__

'1.5.5'

In [82]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

In [83]:
def modify_date(date):
    return date[:10]

In [84]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [85]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [86]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [87]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [88]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [89]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [90]:
# Preselection parameters set up
#method = "K_means_multiple"# or avg or anyother method
method = "K_means_multiple"
stock_type = "S&P100"
end_swan = "Swan9"         #equals the last swan before the swan we are going to predict
start_swan="Swan5"
target = "Swan10d"
start = dt(2020,11,1)
end = dt(2022,1,3)
selected_stock_number = 40     #number of stock preselected
start_swan_date = dt(2022,1,3)
end_swan_date = dt(2023,10,18)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means
Min_appear = 0
Seeds = 5

In [91]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
    #portfolio_beta = pd.read_excel("BetaMatrix S&P100 Cathy(Yijue) 2022.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("SP500_BETALouise 8_30_24 (2).xlsx",index_col=0)

In [92]:
optdata = pd.read_excel("optdata_SP100_20220101_a.xlsx",index_col=0)
portfolio_beta = portfolio_beta[portfolio_beta.index.isin(optdata.index)]

In [93]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [94]:
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10d
Ticker,,,,,,,,,,,,,,
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,NaN,NaN,-0.076048,0.180670,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,NaN,NaN,NaN,-0.071253,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,1.101894,0.665427,-0.825250,1.115449,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
JPM,NaN,NaN,0.873322,1.666279,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,2.119191,1.270985,0.770965,1.317948,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,NaN,NaN,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [95]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [96]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9
Swan1,1.0,0.005966,0.100183,0.060142,-0.001884,0.080724,0.093433,0.066030,0.006523
Swan2,NaN,1.000000,0.592644,0.352313,0.439425,0.467532,0.015261,0.353742,0.078242
Swan3,NaN,NaN,1.000000,0.155481,0.281384,0.415517,0.230374,0.197810,0.070310
Swan4,NaN,NaN,NaN,1.000000,-0.010387,0.180491,0.103724,0.229938,0.150250
Swan5,NaN,NaN,NaN,NaN,1.000000,0.454822,0.301979,0.209499,0.591261
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.378955,0.479729,0.363796
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.152595,0.238611
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.183216
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [97]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [98]:
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10d
Ticker,,,,,,,,,,,,,,
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,NaN,NaN,-0.076048,0.180670,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,NaN,NaN,NaN,-0.071253,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,1.101894,0.665427,-0.825250,1.115449,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
JPM,NaN,NaN,0.873322,1.666279,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,2.119191,1.270985,0.770965,1.317948,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,NaN,NaN,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [99]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,Seeds):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    stock_frequency_name = f"Astock_frequency_w1_{w1}_Seeds_{Seeds}.csv"

    for p in range(0,min(selected_stock_number,len(sorted_count))):
      if sorted_count[p][1]>=Min_appear:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

elif method == "Custom":
    # Step 1: Calculate Davies-Bouldin index for random seeds
    selected_beta = portfolio_beta.loc[:, start_swan:end_swan].dropna()
    dbi_scores = []

    # Loop through random seeds from 1 to 1000 to find the best seeds with lowest DBI
    for R in range(1, 1001):
        kmeans = KMeans(n_clusters=N, random_state=R)
        labels = kmeans.fit_predict(selected_beta)
        dbi = davies_bouldin_score(selected_beta, labels)
        dbi_scores.append((R, dbi))

    # Sort by Davies-Bouldin index and select the top 5 seeds with lowest DBI
    top_seeds = sorted(dbi_scores, key=lambda x: x[1])[:Seeds]


    stock_counter = Counter()

    for R, _ in top_seeds:

        selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
        Y_predic = selected_beta[end_swan].copy()
        Kmean = KMeans(n_clusters=N,random_state=R)#set the random_state to keep modeling the same every run
        Kmean.fit(selected_beta)
        ct = Kmean.cluster_centers_
        Kmean.labels_
        selected_beta["Group"]=Kmean.labels_
        res = ct.mean(axis=1)
        res [res<0]=0

        min_group = np.where(res==min(res))
        for i in range(N):
            Y_predic[selected_beta["Group"] == i] = res [i]
        selected_beta ["avg"]=Y_predic.values
        selected_beta=selected_beta.sort_values(by=["avg"])
        first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
        if first_group.shape[0] <10:
            second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
            preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=True)
        else:
            preselected_stock = first_group

        stock_counter.update(preselected_stock.index)

    top_stocks = [stock for stock, count in stock_counter.most_common(22)]

['SO', 'MCD', 'MDLZ', 'MRK', 'SBUX', 'LLY', 'PG', 'PEP', 'EXC', 'VZ', 'NEE', 'DUK', 'KO', 'AMT', 'SO', 'ABT', 'AMT', 'MCD', 'MDLZ', 'SBUX', 'CL', 'PEP', 'LLY', 'MRK', 'KO', 'WMT', 'EXC', 'VZ', 'DUK', 'PG', 'NEE', 'SO', 'MCD', 'AMT', 'PEP', 'LLY', 'MRK', 'KO', 'PG', 'EXC', 'SBUX', 'NEE', 'DUK', 'SPG', 'MDLZ', 'SO', 'MCD', 'MDLZ', 'MRK', 'SBUX', 'LLY', 'PG', 'PEP', 'EXC', 'VZ', 'NEE', 'DUK', 'KO', 'AMT', 'SO', 'MCD', 'AMT', 'PEP', 'LLY', 'MRK', 'KO', 'PG', 'EXC', 'SBUX', 'NEE', 'DUK', 'SPG', 'MDLZ']
[('AMT', 5), ('DUK', 5), ('EXC', 5), ('KO', 5), ('LLY', 5), ('MCD', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 3), ('SPG', 2), ('ABT', 1), ('CL', 1), ('WMT', 1)]
           Swan5     Swan6     Swan7     Swan8     Swan9  Group       avg
Ticker                                                                   
ABT     0.474695  0.311080  1.073151  0.431920  0.941927      1  0.646555
AMT     0.583652  0.050751  0.583637 -0.336659  0.882155     

<ipython-input-99-54239dd07b2a>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-99-54239dd07b2a>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-99-54239dd07b2a>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

In [100]:
ct

array([[ 1.32070774,  1.42534032,  1.12526335,  1.18748508,  1.46779408],
       [ 0.49399445,  0.15407183,  0.75913122,  0.80849184,  0.65674495],
       [ 0.82930335,  0.87736275,  0.76064394,  0.91592729,  0.92794914],
       [ 0.63571697,  1.25106214, -0.25339946,  2.78743065,  1.0267475 ],
       [ 1.86416708,  2.35824737,  1.28963209,  1.22430697,  1.66153116],
       [ 0.79137378,  0.26777892,  0.51971175, -0.08684112,  1.04290484],
       [-0.5135618 ,  3.47395849,  0.61286881,  1.90242394,  0.21285984]])

In [101]:
res

array([1.30531812, 0.57448686, 0.86223729, 1.08951156, 1.67957694,
       0.50698564, 1.13770986])

In [102]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, ABT to WMT
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan5   18 non-null     float64
 1   Swan6   18 non-null     float64
 2   Swan7   18 non-null     float64
 3   Swan8   18 non-null     float64
 4   Swan9   18 non-null     float64
 5   Group   18 non-null     int32  
 6   avg     18 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.1+ KB


In [103]:
#order = list(preselected_stock.index)
#random.shuffle(order)
#preselected_stock = preselected_stock.reindex(index = order)
preselected_stock


,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg
Ticker,,,,,,,
ABT,0.474695,0.311080,1.073151,0.431920,0.941927,1,0.646555
AMT,0.583652,0.050751,0.583637,-0.336659,0.882155,5,0.352707
CL,0.617541,-0.045655,0.784583,0.569564,0.632697,1,0.511746
DUK,0.771715,-0.322795,0.233695,-0.356252,1.165454,5,0.298364
EXC,0.794675,0.263392,0.605247,-0.123437,1.254518,5,0.558879
KO,0.536675,0.325968,0.614677,-0.049527,1.161009,5,0.517760
LLY,0.920577,0.280674,0.780346,0.061464,0.490481,5,0.506708
MCD,0.321830,0.130073,0.483323,-0.202954,1.142392,5,0.374933
MDLZ,0.828314,0.344207,0.089728,0.386481,0.928976,5,0.515541


In [104]:
#holt step
portfolio_beta = pd.read_csv("HoltestimateS10d.csv",index_col=0)
if 'Ticker' in preselected_stock.columns:
    preselected_stock.set_index('Ticker', inplace=True)
preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan10d'])

In [105]:
#using average step
'''
preselected_stock

preselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5
'''

"\npreselected_stock\n\npreselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5\n"

In [106]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg,PR
Ticker,,,,,,,,
ABT,0.474695,0.311080,1.073151,0.431920,0.941927,1,0.963146,0.036854
AMT,0.583652,0.050751,0.583637,-0.336659,0.882155,5,0.415581,0.584419
CL,0.617541,-0.045655,0.784583,0.569564,0.632697,1,0.705449,0.294551
DUK,0.771715,-0.322795,0.233695,-0.356252,1.165454,5,0.524570,0.475430
EXC,0.794675,0.263392,0.605247,-0.123437,1.254518,5,0.718736,0.281264
KO,0.536675,0.325968,0.614677,-0.049527,1.161009,5,0.779753,0.220247
LLY,0.920577,0.280674,0.780346,0.061464,0.490481,5,0.182871,0.817129
MCD,0.321830,0.130073,0.483323,-0.202954,1.142392,5,0.767374,0.232626
MDLZ,0.828314,0.344207,0.089728,0.386481,0.928976,5,1.471470,-0.471470


# Step 2 - Download data and adjuste price based on Swan Beta

In [107]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [108]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['ABT' 'AMT' 'CL' 'DUK' 'EXC' 'KO' 'LLY' 'MCD' 'MDLZ' 'MRK' 'NEE' 'PEP'
 'PG' 'SBUX' 'SO' 'SPG' 'VZ' 'WMT']


In [109]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]

[*********************100%%**********************]  18 of 18 completed


In [110]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

Ticker,ABT,AMT,CL,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,SPG,VZ,WMT
Date,,,,,,,,,,,,,,,,,,
2020-11-02,100.058777,209.372604,72.979820,79.120132,25.136732,42.729202,125.202492,192.901428,49.554581,64.807381,67.856613,119.612961,125.442467,78.484238,49.966709,51.310570,45.706928,43.983192
2020-11-03,101.995148,210.036118,75.665573,80.553772,26.063406,43.370758,124.584213,196.749283,49.627533,64.984795,68.660431,121.285706,127.906006,80.693512,50.686455,51.532589,45.746536,44.728779
2020-11-04,103.829132,212.582779,76.348404,78.892433,25.909986,43.212563,141.334320,194.997787,50.101738,68.110695,66.221832,122.011429,127.380684,81.971619,50.601772,49.383751,45.326698,44.471878
2020-11-05,105.691025,215.138412,75.574539,79.457466,26.431623,43.449860,138.414230,196.304611,50.740089,67.975525,67.513412,122.179604,128.956650,82.729347,50.948940,50.327335,46.063396,44.944935
2020-11-06,106.519569,217.138046,75.592735,78.799660,26.290470,43.458645,135.770035,196.531494,50.922489,67.891060,68.389496,122.471619,129.726517,82.756721,50.771126,49.050720,46.364410,45.665459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,133.728134,262.766388,78.139267,91.466377,35.100742,53.585857,270.209961,250.485428,61.056480,70.810928,84.496925,157.304153,150.389557,106.497963,59.684811,133.305939,43.621742,44.960949
2021-12-28,132.792236,265.276703,78.651436,92.599869,35.613068,53.796001,268.288269,250.149307,61.513096,71.032883,85.015076,158.120605,151.197311,107.756683,60.371151,132.750595,43.654865,45.606167
2021-12-29,133.472885,265.635376,79.070488,92.768120,36.279099,53.859951,270.219696,250.718887,61.848572,71.162354,85.116867,158.680252,152.450806,108.511925,60.674210,133.583588,43.356770,45.583813


In [111]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [112]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [113]:
market_drop.to_csv("market_drop.csv")

In [114]:
market_drop

,0
0,0.0000
1,0.0044
2,0.0088
3,0.0132
4,0.0176
...,...
289,0.0000
290,0.0000
291,0.0000
292,0.0000


In [115]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [116]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMT', 'CL', 'DUK', 'EXC', 'KO', 'LLY', 'MCD', 'MDLZ',
       'MRK', 'NEE', 'PEP', 'PG', 'SBUX', 'SO', 'SPG', 'VZ', 'WMT'],
      dtype=object)

In [117]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame

Ticker,ABT,AMT,CL,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,SPG,VZ,WMT
Date,,,,,,,,,,,,,,,,,,
2020-11-02,100.058777,209.372604,72.979820,79.120132,25.136732,42.729202,125.202492,192.901428,49.554581,64.807381,67.856613,119.612961,125.442467,78.484238,49.966709,51.310570,45.706928,43.983192
2020-11-03,101.562908,209.652055,75.430709,80.367845,25.980982,43.221957,124.483969,196.084970,49.306221,64.903285,68.469198,121.043168,127.738764,80.571354,50.292489,50.691178,45.670253,44.678073
2020-11-04,102.949109,211.805340,75.874437,78.528249,25.746109,42.916045,141.106876,193.680989,49.452974,67.939835,65.852950,121.523450,127.047574,81.723432,49.815157,47.771100,45.175531,44.371050
2020-11-05,104.347319,213.958234,74.870795,78.907277,26.180858,43.002641,138.080112,194.316176,49.754544,67.719743,66.949297,121.446627,128.450802,82.353626,49.760922,47.862139,45.832961,44.792084
2020-11-06,104.713917,215.549850,74.654184,78.072147,25.957902,42.862233,135.333055,193.877182,49.603705,67.550442,67.627582,121.491981,129.048027,82.255594,49.192630,45.847169,46.055155,45.458390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,133.728134,262.766388,78.139267,91.466377,35.100742,53.585857,270.209961,250.485428,61.056480,70.810928,84.496925,157.304153,150.389557,106.497963,59.684811,133.305939,43.621742,44.960949
2021-12-28,132.792236,265.276703,78.651436,92.599869,35.613068,53.796001,268.288269,250.149307,61.513096,71.032883,85.015076,158.120605,151.197311,107.756683,60.371151,132.750595,43.654865,45.606167
2021-12-29,133.472885,265.635376,79.070488,92.768120,36.279099,53.859951,270.219696,250.718887,61.848572,71.162354,85.116867,158.680252,152.450806,108.511925,60.674210,133.583588,43.356770,45.583813


# Step3 - MVO

In [118]:
#Set the paramter for MVO process
Portfolio_Date=end

In [119]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [120]:
#calculated rf
#bond_yield = pd.read_csv("bond_yield.csv", index_col=0)
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [121]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

In [122]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [123]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [124]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [125]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [126]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [127]:
# Read OPT-data
optdata = pd.read_excel("optdata_SP100_20220101_a.xlsx",index_col=0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [128]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [129]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)

/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [130]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [131]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)

OrderedDict([('ABT', 0.04743), ('AMT', 0.0), ('CL', 0.0), ('DUK', 0.02545), ('EXC', 0.17), ('KO', 0.0), ('LLY', 0.06892), ('MCD', 0.11199), ('MDLZ', 0.0), ('MRK', 0.05122), ('NEE', 0.0), ('PEP', 0.0), ('PG', 0.16162), ('SBUX', 0.02338), ('SO', 0.0), ('SPG', 0.0), ('VZ', 0.17), ('WMT', 0.17)])
Expected annual return: 4.9%
Annual volatility: 10.2%
Sharpe Ratio: 0.38


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.04856845723295648, 0.1020453394422195, 0.37795412748658513)

In [132]:
cleaned_weights

OrderedDict([('ABT', 0.04743),
             ('AMT', 0.0),
             ('CL', 0.0),
             ('DUK', 0.02545),
             ('EXC', 0.17),
             ('KO', 0.0),
             ('LLY', 0.06892),
             ('MCD', 0.11199),
             ('MDLZ', 0.0),
             ('MRK', 0.05122),
             ('NEE', 0.0),
             ('PEP', 0.0),
             ('PG', 0.16162),
             ('SBUX', 0.02338),
             ('SO', 0.0),
             ('SPG', 0.0),
             ('VZ', 0.17),
             ('WMT', 0.17)])

In [133]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [134]:
weight1

array([0.04743, 0.     , 0.     , 0.02545, 0.17   , 0.     , 0.06892,
       0.11199, 0.     , 0.05122, 0.     , 0.     , 0.16162, 0.02338,
       0.     , 0.     , 0.17   , 0.17   ])

In [135]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.3779525719764981


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [136]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [137]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed


In [138]:
stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,ABT,EXC,LLY,MCD,MRK,PG,VZ,WMT
Date,,,,,,,,
2022-01-03,131.440414,36.548065,263.726715,250.802917,71.088364,151.253067,43.423012,46.203476
2022-01-04,128.349121,36.509640,258.951630,250.737610,71.217834,151.782288,44.275902,45.357029
2022-01-05,127.772484,36.484024,252.973038,249.504944,72.947189,152.469360,44.731331,45.970306
2022-01-06,127.753548,36.330326,251.691910,251.839523,72.900948,151.188034,44.516029,45.842541
2022-01-07,128.150604,36.272694,251.856949,249.383591,74.260384,151.104446,45.454632,46.280140
...,...,...,...,...,...,...,...,...
2023-10-11,90.327415,38.081081,600.246155,243.524048,100.787491,138.651321,29.494606,51.925774
2023-10-12,88.425377,37.881359,605.422729,238.933472,100.914131,138.961578,28.978966,52.162056
2023-10-13,89.092072,37.995491,604.133484,240.990967,101.323296,140.289902,28.753960,52.450848


In [139]:
weight

,weight
ABT,0.049864
EXC,0.178725
LLY,0.072457
MCD,0.117738
MRK,0.053849
PG,0.169915
VZ,0.178725
WMT,0.178725


In [140]:
start=stock_price.loc["2022-01-04"]
end = stock_price.loc["2022-10-12"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2022-10-12"]/OEX.loc["2022-01-04"]-1)

In [141]:
beta = stock_drop/market_drop
1-beta.T@weight

,0
weight,0.362134


In [142]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>w1:
    rest=rest-row['weight']
    extra=extra+(row['weight']-w1)
    row['weight']=w1

for i,row in weight.iterrows():
  if row['weight']<(w1-0.05):
    row['weight']=row['weight']+row['weight']/rest*extra
    a=1


weights_name = f"Aweights_w1_{w1}_Seeds_{Seeds}.csv"
weight

,weight
ABT,0.052678
EXC,0.170000
LLY,0.076547
MCD,0.124383
MRK,0.056888
PG,0.169915
VZ,0.170000
WMT,0.170000


In [143]:
portfolio_beta = portfolio_beta.loc[weight.index]

swan_beta_name = f"A_forcast_and_actual_Beta{w1}_Seeds_{Seeds}.csv"

portfolio_beta

,Forecasted Swan10d,Actual Swan10d
ABT,0.963146,0.997309
EXC,0.718736,0.353268
LLY,0.182871,-0.761718
MCD,0.767374,0.376953
MRK,0.285064,-0.756863
PG,0.297168,0.817893
VZ,0.378983,1.036945
WMT,0.257641,0.302591


In [144]:
a=stock_price@weight
b=1-a.loc["2022-10-12"]/a.loc["2022-01-04"]
1-b/market_drop

,0
weight,0.705636


In [145]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [146]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX

<ipython-input-146-1f2d32375bd9>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])


In [147]:
temp=injected_result.loc[["2022-01-04","2022-10-12"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.5738765922156647


<ipython-input-147-d8f559932dfb>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-147-d8f559932dfb>:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2022-01-04,9884.975130,9969.541822
2022-10-12,8747.359181,7277.015862


In [148]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-88-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-88-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-88-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
WMT,0.170000,2.477669,11.139314,0.125,0.389762,0.17,0.17
VZ,0.170000,0.925932,2.22306,0.125,0.145658,0.17,0.17
PG,0.169915,0.805699,1.314143,0.125,0.126744,0.159315,0.17
MRK,0.056888,0.688423,0.927207,0.125,0.108296,0.140866,0.17
LLY,0.076547,0.682153,0.696919,0.125,0.107309,0.139879,0.17
EXC,0.170000,0.50676,3.246214,0.125,0.079718,0.112288,0.05
MCD,0.124383,0.233824,1.010324,0.125,0.036783,0.069353,0.05
ABT,0.052678,0.036418,0.976449,0.125,0.005729,0.038299,0.05


In [149]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-89-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-89-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 comple

#

In [150]:
Portfolio_DD_L = abs(injected_result['portfolio_value'].iloc[195] / injected_result['portfolio_value'].iloc[1] - 1)
DD_L = abs(injected_result['oex'].iloc[195] / injected_result['oex'].iloc[1] - 1)
PR_L = 1 - (Portfolio_DD_L / DD_L)

print(f"Portfolio Drawdown: {Portfolio_DD_L}")
print(f"DD: {DD_L}")
print(f"Performance Ratio (PR): {PR_L}")

Portfolio Drawdown: 0.11508536279997084
DD: 0.27007519581796013
Performance Ratio (PR): 0.5738765922156647


In [151]:
injected_result['oex'].iloc[1]

9969.541821612249

In [152]:
PR = pd.DataFrame([PR_L, DD_L, Portfolio_DD_L], columns=['PR and DD'])
PR.index = ['PR', 'MarketDD', 'PortfolioDD']
PR

,PR and DD
PR,0.573877
MarketDD,0.270075
PortfolioDD,0.115085


In [153]:
injected_result_name = f"AInjected_result_w1_{w1}_Seeds_{Seeds}.csv"

In [154]:
gradient_matrix = weight.join(portfolio_beta, how='outer')

In [155]:
file_name = f"AA_{target}_gradient_matrix_Weight:_{w1}_Seeds:_{Seeds}_K:_{N}.xlsx"

with pd.ExcelWriter(file_name) as writer:
    gradient_matrix.to_excel(writer, sheet_name='Sheet1 - Weights and Beta')
    injected_result.to_excel(writer, sheet_name='Sheet2 - injected_result')
    df.to_excel(writer, sheet_name='Sheet3 - frequency')
    PR.to_excel(writer, sheet_name='Sheet4 - PR_rate')